<a href="https://colab.research.google.com/github/todd-jang/AIFFEL_quest_rs/blob/main/MainQuest/Quest04/GENAI_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [3]:
import os


In [4]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [5]:
#! curl ipinfo.io

In [6]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [7]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.

https://reference.langchain.com/python/langchain-community/document_loaders

https://docs.langchain.com/oss/python/langchain/overview

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [8]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_26829/128205925.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [9]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [10]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [11]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [12]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/titanic.csv")

data = loader.load()

In [13]:
data[:3]

[Document(metadata={'source': '/content/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [14]:
from langchain_community.document_loaders import WebBaseLoader

In [68]:
loader = WebBaseLoader("https://www.pado.kr/article/2026092513108817987")
documents = loader.load()

print(documents[0].page_content)

'남성의 외로움 위기'가 착각인 까닭 | PADOSearchMyPageMagazine전체보기국제정세경제사회과학문학예술FeatureBooks'남성의 외로움 위기'가 착각인 까닭보다 고립되고 외로운 사람들은 분명 존재하지만 그게 전부 남성은 아니다2026.09.25 16:15Slate20min0/그래픽=Tolga Akdoğan편집자주슬레이트 매거진의 8월 13일자 에세이는 '미국 남성의 외로움'이라는 화두가 외로움의 본질을 호도하는 잘못된 화두임을 지적합니다. 우리도 남자들은 사교성이 부족하고 평소 친구들도 별로 없는 것처럼, 그리고 이에 반해 여성들은 타인들과 쉽게 소통하고 늘 전화를 할 친구들이 있는 것처럼 이야기합니다. 하지만, 실제 통계를 보면 외로움, 고립은 성별보다는 오히려 소득과 더 관련되어 있습니다. 즉, 소득이 높은 사람들은 사교활동이 활발하고 낮은 사람들은 그렇지 않다는 것입니다. 단적인 사실로, 사교활동에는 비용이 듭니다. 친구와 커피 한 잔을 하더라도 커피 값이 필요합니다. 좀 더 길게 만나려면 식사비용이 듭니다. 이성과 데이트를 하려면 상당한 비용이 듭니다. 이것이 여의치 않은 저소득층은 사교를 '선제적으로' 피합니다. 미국 사회는 경제적 양극화가 심해지고 있습니다. 이와 함께 '사교'의 양극화도 심해질 수밖에 없습니다. 이런 사교의 양극화는 비단 미국만의 문제가 아닙니다. 한국도 마찬가지입니다. '사교에는 돈이 필요하다'는 것은 어디서나 마찬가지이기 때문입니다. 외로움과 빈곤의 상관관계를 지적한 이 에세이는 남성들의 젠더적 특성이나 개인적 성격으로 구조적 문제를 환원하는 문제를 지적합니다. 사회적 동물인 인간이 가장 두려워하는 것은 외로움, 고립감입니다. 눈이 멀고 귀가 머는 것을 두려워하는 것도 그것이 주는 고립감이 두려워서입니다. 죽음을 두려워하는 것 역시 죽음이 가져올 '절대적 외로움' 때문입니다. 우리는 고립되어 외로워하는 사람들을 챙겨야합니다. 외로움은 개인을 파괴하고 결국 사회를 파괴하기 때문입니다.남자들이 외롭다는 이야기, 한

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [16]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [17]:
with open("/content/state_of_the_union.txt") as f:
    text = f.read()

In [18]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [19]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [20]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [21]:
!pip install tiktoken

In [22]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [23]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !

### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [24]:
import openai

genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [25]:
client = openai.OpenAI()

In [26]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-3-small
text-embedding-3-large
text-embedding-ada-002


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [27]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [28]:
#!curl ipinfo.io

In [29]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [30]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [31]:
print(embeddings[1])

[0.006381988525390625, -0.0208740234375, -0.0216217041015625, 0.0235595703125, -0.042877197265625, -0.004638671875, 0.039703369140625, 0.05267333984375, -0.0019063949584960938, -0.023773193359375, 0.0137939453125, 0.0159759521484375, -0.044464111328125, -0.00013434886932373047, -0.0076751708984375, 0.02557373046875, 0.025360107421875, 0.035491943359375, -0.051727294921875, 0.012359619140625, 0.02484130859375, -0.0004699230194091797, 0.019378662109375, 0.07525634765625, -0.0020751953125, -0.00919342041015625, 0.0168304443359375, -0.00713348388671875, 0.02655029296875, -0.048492431640625, 0.043182373046875, -0.04254150390625, 0.01165008544921875, -0.0328369140625, -0.033294677734375, -0.046142578125, 0.0019016265869140625, 0.0225830078125, -0.01812744140625, 0.03253173828125, 0.0293121337890625, 0.0117340087890625, -0.01439666748046875, 0.0030651092529296875, 0.02435302734375, 0.04815673828125, -0.01357269287109375, 0.04931640625, -0.040679931640625, 0.04400634765625, 0.045196533203125, 

In [32]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [33]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [34]:
query = ["this is red fruit"]

In [35]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7479283690067156
0.4898695305933273
0.4083756794905314


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [36]:
!pip install chromadb

In [37]:
!pip install langchain-chroma

In [38]:
from langchain_chroma import Chroma

In [39]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [40]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [41]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [42]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [43]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [44]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [45]:
!pip install -U langchain langchain-classic

In [46]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [47]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [48]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [49]:
query = "how demian looks like"
result = qa(query)

/tmp/ipykernel_26829/3336337621.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  result = qa(query)


Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. It is suggested that his appearance is unique, almost like an animal, a spirit, or an image, making him unimaginably different from others.

마크다운 형식으로 출력해봅니다

In [50]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. It is suggested that his appearance is unique, almost like an animal, a spirit, or an image, making him unimaginably different from others.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [51]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you are referring to the character Emil Sinclair from Hermann Hesse's novel "Demian," he is the protagonist and narrates his journey of self-discovery. Emil Sinclair is depicted as a young man searching for his own identity and grappling with the duality of light and darkness within himself. The novel provides more insight into his psychological and spiritual development rather than focusing on detailed physical descriptions. Throughout the story, his interactions with other characters, such as the enigmatic Max Demian, help shape his understanding of the world and his place within it. The emphasis is more on his internal transformation rather than a concrete physical appearance.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [52]:
import os
from langchain_community.document_loaders import TextLoader
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from IPython.display import Markdown, display
import tiktoken

# Download the text of The Merchant of Venice
!wget -nc https://www.gutenberg.org/files/2243/2243-0.txt -O /content/merchant_of_venice.txt

--2026-10-01 05:38:01--  https://www.gutenberg.org/files/2243/2243-0.txt
Resolving www.gutenberg.org (www.gutenberg.org)... 152.19.134.47, 2610:28:3090:3000:0:bad:cafe:47
Connecting to www.gutenberg.org (www.gutenberg.org)|152.19.134.47|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 124897 (122K) [text/plain]
Saving to: ‘/content/merchant_of_venice.txt’

/content/merchant_o 100%[===================>] 121.97K  --.-KB/s    in 0.06s   

2026-10-01 05:38:01 (1.98 MB/s) - ‘/content/merchant_of_venice.txt’ saved [124897/124897]



Text splitter 사용을 위한 준비입니다

In [53]:
tokenizer = tiktoken.get_encoding("cl100k_base")
def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

### Step 1 Document loader

In [54]:
loader = TextLoader("/content/merchant_of_venice.txt")
pages = loader.load()

### Step 2 Text splitters

In [55]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

### Step 3 Vector Empeddings

In [56]:
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

In [57]:
db = Chroma.from_documents(docs, embedding_model)

### Step 4 Retrievers

In [58]:
llm_rag = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
    streaming=True,
    callbacks=[StreamingStdOutCallbackHandler()],
)

In [59]:
qa = RetrievalQA.from_chain_type(llm_rag, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

### Step 5 Question Answering

In [60]:
query = "Describe the main characters Portia and Shylock from 'The Merchant of Venice'."
result = qa(query)

In "The Merchant of Venice," Portia and Shylock are two of the main characters, each with distinct characteristics and roles in the play:

**Portia**: Portia is a wealthy heiress from Belmont, known for her beauty, intelligence, and wit. She is bound by her father's will to marry the man who chooses the correct casket from three options, which adds a layer of complexity to her character. Portia is resourceful and clever, as demonstrated when she disguises herself as a young male lawyer named Balthasar to save Antonio in court. Her actions reveal her as a strong, independent woman who is capable of navigating and influencing the male-dominated world around her.

**Shylock**: Shylock is a Jewish moneylender in Venice and serves as one of the play's antagonists. He is portrayed as both a villain and a victim, driven by a deep-seated grudge against Antonio, a Christian merchant who has mistreated him. Shylock's character is complex; he is depicted as vengeful and obsessed with the bond he 

In [61]:
display(Markdown(result["result"]))

In "The Merchant of Venice," Portia and Shylock are two of the main characters, each with distinct characteristics and roles in the play:

**Portia**: Portia is a wealthy heiress from Belmont, known for her beauty, intelligence, and wit. She is bound by her father's will to marry the man who chooses the correct casket from three options, which adds a layer of complexity to her character. Portia is resourceful and clever, as demonstrated when she disguises herself as a young male lawyer named Balthasar to save Antonio in court. Her actions reveal her as a strong, independent woman who is capable of navigating and influencing the male-dominated world around her.

**Shylock**: Shylock is a Jewish moneylender in Venice and serves as one of the play's antagonists. He is portrayed as both a villain and a victim, driven by a deep-seated grudge against Antonio, a Christian merchant who has mistreated him. Shylock's character is complex; he is depicted as vengeful and obsessed with the bond he has made with Antonio, demanding a pound of flesh as collateral. However, he is also shown as a man who has suffered discrimination and prejudice, which elicits some sympathy from the audience. Shylock's character explores themes of justice, revenge, and mercy.

### RAG 시스템 결과 검토

위 답변은 '베니스의 상인'의 주요 인물인 포샤(Portia)와 샤일록(Shylock)에 대한 묘사입니다. RAG 시스템이 문서에서 관련 정보를 추출하여 요약했는지 확인해보세요.

### 웹페이지 요약 RAG 예제

새로운 웹페이지(`https://www.pado.kr/article/2026092513108817987`)의 내용을 로드하여 RAG 시스템을 통해 요약하는 과정을 살펴보겠습니다.

In [62]:
from langchain_community.document_loaders import WebBaseLoader

# 웹페이지 로드
web_loader = WebBaseLoader("https://www.pado.kr/article/2026092513108817987")
web_pages = web_loader.load()

print(f"Loaded {len(web_pages)} documents from the webpage.")
# print(web_pages[0].page_content[:500]) # 처음 500자 확인

Loaded 1 documents from the webpage.


In [63]:
# 텍스트 분할 (기존 tiktoken_len 함수와 RecursiveCharacterTextSplitter 사용)
web_text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
web_docs = web_text_splitter.split_documents(web_pages)

print(f"Split into {len(web_docs)} chunks.")

Split into 35 chunks.


In [64]:
# ChromaDB를 사용하여 새로운 벡터 저장소 생성
# 기존 embedding_model 사용
web_db = Chroma.from_documents(web_docs, embedding_model)

print("Vector store created for the webpage.")

Vector store created for the webpage.


In [77]:
# Retriever 설정 (기존 llm_rag 사용)
web_qa = RetrievalQA.from_chain_type(llm_rag, chain_type="stuff",
                                 retriever=web_db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

# 웹페이지 내용 요약 질의
web_query = "Summarize the content of this webpage."
web_result = web_qa(web_query)

The webpage contains excerpts from two different texts. The first excerpt is from "Demian" by Hermann Hesse, where the narrator discusses conversations with Pistorius about human potential, dreams, and the feeling of connection with power. It also touches on societal issues and the potential for future upheaval. The second excerpt is from a Project Gutenberg text of Shakespeare's "The Merchant of Venice," specifically the first act, with notes on the transcription process and textual variations in different editions of the First Folio.

In [66]:
# 결과 출력
display(Markdown(web_result["result"]))

The webpage contains excerpts from two different texts. The first excerpt is from "Demian" by Hermann Hesse, where the narrator discusses conversations with Pistorius about human potential, dreams, and the feeling of connection with power. It also touches on societal issues and the potential for future upheaval. The second excerpt is from a Project Gutenberg text of Shakespeare's "The Merchant of Venice," specifically the first act, with notes on the transcription process and textual variations in different editions of the First Folio.

### 한자 고문서 RAG 파이프라인 예제

한자로 된 고문서를 처리하기 위한 RAG 파이프라인을 구축합니다. 이 과정에는 OCR을 통한 텍스트 추출, 적절한 임베딩 모델 사용, 그리고 기존 RAG 시스템 구성이 포함됩니다.

#### 1. OCR 라이브러리 설치 및 준비

한자 고문서는 대부분 이미지 형태로 되어 있을 것이므로, 먼저 OCR(Optical Character Recognition)을 사용하여 이미지에서 텍스트를 추출해야 합니다. 여기서는 `EasyOCR` 라이브러리를 사용하며, 한국어 및 중국어 모델을 로드하여 사용합니다. (설치에 시간이 다소 소요될 수 있습니다.)

In [78]:
!pip install -q easyocr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 31.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.4/183.4 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 972.1/972.1 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 21.0 MB/s eta 0:00:00


### 고문서 OCR 및 텍스트 추출의 어려움 설명

광개토대왕비문과 같이 오래되고 손상된 고문서 이미지에서 정확하게 OCR(광학 문자 인식)을 수행하고 텍스트를 추출하는 것은 매우 어려운 작업입니다.

1.  **고문서 OCR의 난이도**: 고문서는 일반적으로 종이의 노화, 훼손, 잉크 번짐, 희미해진 글자, 비정형적인 서체 등 다양한 손상을 포함하고 있습니다. 이러한 손상된 이미지에서 텍스트를 정확하게 인식하는 것은 현대의 깨끗한 문서에 비해 훨씬 복잡하고 높은 오류율을 보입니다. 일반적인 OCR 기술은 정형화된 글꼴과 깨끗한 배경을 전제로 학습된 경우가 많아, 고문서의 불규칙성과 다양성을 처리하는 데 한계가 있습니다.

2.  **EasyOCR 설정의 한계**: 현재 EasyOCR의 설정이 'ch_sim'(중국어 간체)으로 되어 있는데, 이는 한자 고문서에 적합하지 않을 수 있습니다. 고문서에 사용된 한자는 현대 중국어 간체자와는 다른 번체자 또는 더 오래된 자형을 포함하며, 문체 또한 고어체인 경우가 많습니다. EasyOCR의 기본 모델은 현대적인 글꼴과 언어 패턴에 최적화되어 있어, 고문서 특유의 자형과 문맥을 정확하게 인식하기 어렵습니다. 따라서 고문서 OCR의 정확도를 높이기 위해서는 해당 시대의 서체와 언어에 맞춰 특별히 훈련된 모델이 필요합니다.

3.  **정확도 향상을 위한 방안**: 고문서 OCR의 정확도를 높이기 위해서는 다음 단계들이 필수적입니다.
    *   **이미지 전처리**: 노이즈 제거, 대비 조절, 글자 강조 등 손상된 이미지를 개선하는 전처리 과정이 중요합니다.
    *   **전문 OCR 모델**: 특정 시대와 서체에 맞춰 훈련된 전문 OCR 모델을 사용하거나, 기존 모델을 고문서 데이터로 추가 학습(Fine-tuning)하는 방식이 필요합니다.

In [84]:
import easyocr
import os

# EasyOCR 리더 초기화 (중국어 간체, 영어 지원)
# 처음 실행 시 언어 모델 다운로드가 필요할 수 있습니다.
# 오류 메시지: "Chinese_sim is only compatible with English, try lang_list=["ch_sim","en"]"
ocr_reader = easyocr.Reader(['ch_sim', 'en'], gpu=True) # gpu=True는 GPU 사용 시

# 예시 이미지 파일 경로 (실제 고문서 이미지 파일로 교체 필요)
# 이 파일을 Colab 환경에 업로드하거나 Drive에서 불러와야 합니다.
ancient_document_image_path = '/content/ancient_document_example.png'

# 가상의 이미지 파일 생성 (실제 사용 시에는 사용자의 이미지 파일 경로를 사용)
# 실제 고문서 이미지 파일을 이 경로에 두어야 합니다.
# 예: ancient_document_image_path = '/content/drive/MyDrive/고문서.png'

# 여기서는 임시로 빈 파일을 생성하여 오류를 방지하지만, 실제로는 유효한 이미지여야 합니다.
# with open(ancient_document_image_path, 'w') as f:
#     f.write('')

# OCR을 통해 텍스트 추출 (주석 처리됨: 실제 이미지 파일이 있어야 실행 가능)
# try:
#     ocr_results = ocr_reader.readtext(ancient_document_image_path)
#     extracted_text = " ".join([res[1] for res in ocr_results])
#     print("OCR 추출 텍스트의 일부:", extracted_text[:500])
# except Exception as e:
#     extracted_text = "" # 오류 발생 시 빈 문자열
#     print(f"OCR 처리 중 오류 발생: {e}. 실제 이미지 파일이 필요합니다.")

# 임시로 고문서와 유사한 텍스트를 가정합니다.
# 사용자는 이 부분을 실제 OCR 결과로 대체해야 합니다.
extracted_text = "夫子曰 學而時習之 不亦說乎 有朋自遠方來 不亦樂乎 人不知而不慍 不亦君子乎" # 논어 구절 예시
print("가정된 고문서 텍스트:", extracted_text)

Progress: |██████████████████████████████████████████████████| 100.0% Complete가정된 고문서 텍스트: 夫子曰 學而時習之 不亦說乎 有朋自遠方來 不亦樂乎 人不知而不慍 不亦君子乎


#### 2. 추출된 텍스트 로드 및 분할

OCR로 추출된 텍스트를 `Document` 객체로 로드하고, `RecursiveCharacterTextSplitter`를 사용하여 청크로 분할합니다. 한자 텍스트의 특성을 고려하여 `tiktoken_len` 함수를 활용합니다.

In [86]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# 임시로 고문서와 유사한 텍스트를 가정합니다.
# 사용자는 이 부분을 실제 OCR 결과로 대체해야 합니다.
extracted_text = "夫子曰 學而時習之 不亦說乎 有朋自遠方來 不亦樂乎 人不知而不慍 不亦君子乎" # 논어 구절 예시

# 추출된 텍스트를 파일로 저장 (LangChain TextLoader 사용을 위해)
# 실제 사용 시에는 OCR 결과를 직접 TextLoader에 전달하거나, 메모리에 로드된 텍스트를 사용합니다.
with open("/content/ancient_doc.txt", "w", encoding="utf-8") as f:
    f.write(extracted_text)

# 텍스트 로더를 사용하여 Document 객체로 로드
ancient_loader = TextLoader("/content/ancient_doc.txt", encoding="utf-8")
ancient_pages = ancient_loader.load()

# 텍스트 분할 (한자 특성 고려)
ancient_text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50, length_function=tiktoken_len)
ancient_docs = ancient_text_splitter.split_documents(ancient_pages)

print(f"고문서 텍스트를 {len(ancient_docs)}개의 청크로 분할했습니다.")

고문서 텍스트를 1개의 청크로 분할했습니다.


#### 3. 한자 임베딩 모델 준비 및 벡터 저장소 생성

한자 텍스트에 적합한 임베딩 모델을 사용해야 합니다. 여기서는 범용적인 HuggingFace 임베딩 모델을 예시로 사용하지만, 더 정교한 한자 또는 고전 중국어 임베딩 모델을 찾아 적용할 수 있습니다. 이후 `Chroma` 벡터 저장소를 생성합니다.

In [87]:
# 한자 처리에 적합한 임베딩 모델 사용 (예: 다국어 모델)
# OpenAI embedding_model은 영어에 최적화되어 있을 수 있습니다.
# !pip install -q sentence_transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

# 다국어 임베딩 모델 사용 (예시)
hanja_embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# ChromaDB를 사용하여 벡터 저장소 생성 (고유한 컬렉션 이름 지정)
hanja_db = Chroma.from_documents(ancient_docs, hanja_embedding_model, collection_name="ancient_hanja_collection")

print("한자 고문서를 위한 벡터 저장소를 생성했습니다.")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

한자 고문서를 위한 벡터 저장소를 생성했습니다.


#### 4. Retriever 설정 및 질의응답

기존에 설정된 `llm_rag` 모델과 새로운 벡터 저장소를 연결하여 RAG 체인을 구성하고, 한자 고문서에 대한 질문을 시도합니다.

In [92]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA

# Retriever 설정 (기존 llm_rag 사용, 새 벡터 저장소 연결)
hanja_qa = RetrievalQA.from_chain_type(llm_rag, chain_type="stuff",
                                 retriever=hanja_db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

# 한자 고문서에 대한 질의
hanja_query = "위 글에서 강조하는 학문의 태도는 무엇인가요?" # '논어' 구절에 대한 질문 예시
hanja_result = hanja_qa(hanja_query)

위 글에서 강조하는 학문의 태도는 "배우고 때때로 익히는 것"입니다. 이는 학문을 지속적으로 배우고 반복하여 익히는 것이 중요하다는 것을 나타냅니다.

In [93]:
from IPython.display import Markdown, display
display(Markdown(hanja_result["result"]))

위 글에서 강조하는 학문의 태도는 "배우고 때때로 익히는 것"입니다. 이는 학문을 지속적으로 배우고 반복하여 익히는 것이 중요하다는 것을 나타냅니다.

### 한자 고문서 RAG 시스템 결과 검토

위 답변은 한자 고문서에 대한 RAG 시스템의 질의응답 결과입니다. 실제 고문서 내용을 기반으로 올바른 답변이 생성되었는지 확인해 주세요. 정확도를 높이기 위해서는 고문서의 폰트, 이미지 품질, 그리고 사용되는 OCR 및 임베딩 모델의 한자 처리 능력이 매우 중요합니다.

### 현재 고문서 RAG 파이프라인의 활용 가능성 및 다음 단계

현재 구축된 RAG 파이프라인은 OCR을 통해 텍스트로 변환된 고문서 데이터를 성공적으로 처리하여 질문에 답할 수 있습니다. 이전 단계에서 발생했던 `NameError: name 'hanja_db' is not defined` 오류는 `hanja_db` 변수를 올바르게 초기화함으로써 해결되었으며, 이제 파이프라인은 고문서 텍스트를 임베딩하고 검색 기반 질의응답을 수행할 수 있는 상태입니다.

가정된 '논어' 구절을 사용하여 RAG 시스템은 해당 텍스트의 내용을 이해하고 관련 질문에 대한 답변을 생성하는 능력을 보여주었습니다. 이는 향후 실제 고문서 연구 및 활용에 RAG 시스템을 적용할 수 있는 기반을 마련합니다.

하지만 실제 광개토대왕비문과 같이 손상된 고문서 이미지에서 텍스트를 정확하게 추출하는 것은 여전히 큰 과제입니다. OCR 정확도를 높이기 위해서는 다음 단계들이 필수적입니다:

1.  **이미지 전처리**: 고문서 이미지의 노이즈 제거, 대비 조절, 글자 강조 등의 전처리 작업을 통해 OCR 성능을 향상시켜야 합니다.
2.  **전문 OCR 모델**: 현대 한자 텍스트에 최적화된 EasyOCR의 'ch_sim' 모델 대신, 고문서의 번체자 및 고어체 특성에 맞춰 특별히 훈련된 전문 OCR 모델을 사용하거나 개발해야 합니다.
3.  **Fine-tuning**: 필요에 따라 기존 OCR 모델을 고문서 데이터셋으로 추가 학습(Fine-tuning)하여 특정 고문서의 서체와 언어 패턴에 대한 인식률을 극대화할 수 있습니다.

이러한 추가 단계를 통해 고문서 RAG 파이프라인의 전반적인 정확도와 신뢰도를 크게 높일 수 있을 것입니다.

### `hanja_db` NameError 해결 및 RAG 시스템 완성

`NameError: name 'hanja_db' is not defined` 오류는 `hanja_db` 변수가 정의되기 전에 사용되었기 때문에 발생합니다. 이는 이전 셀이 실행되지 않았거나 오류로 인해 `hanja_db`가 생성되지 않았을 때 발생할 수 있습니다.

이 문제를 해결하고 한자 고문서 RAG 파이프라인을 완성하려면 다음 단계를 순서대로 실행해주세요:

1.  **셀 `f60a2259` 다시 실행:** `hanja_db`를 생성하는 코드가 포함된 셀을 다시 실행하여 `hanja_db` 변수가 올바르게 정의되도록 합니다. 이 셀은 '한자 처리에 적합한 임베딩 모델 준비 및 벡터 저장소 생성'이라는 제목 아래에 있습니다.

2.  **셀 `dc85b477` 다시 실행:** `hanja_db`가 성공적으로 생성된 것을 확인한 후, `hanja_qa` 체인을 정의하는 셀을 다시 실행합니다. 이 셀은 'Retriever 설정 및 질의응답'이라는 제목 아래에 있습니다.

3.  **셀 `0de03b83` 다시 실행:** 마지막으로, `hanja_query`를 사용하여 RAG 시스템에 질문하고 결과를 출력하는 셀을 다시 실행합니다.

이 단계를 완료하면 `NameError`가 해결되고 한자 고문서 RAG 시스템의 질의응답 결과를 확인할 수 있을 것입니다.

**Reasoning**:
The `NameError` indicates that `hanja_db` was not successfully defined in the previous execution. To fix this, I need to ensure the cell creating `hanja_db` is executed first. I will provide a code block that re-runs the logic from cell `f60a2259` to define `hanja_db`.



In [88]:
print("--- Re-creating Hanja Embedding Model and Vector Store ---")

# 한자 처리에 적합한 임베딩 모델 사용 (예: 다국어 모델)
# OpenAI embedding_model은 영어에 최적화되어 있을 수 있습니다.
# !pip install -q sentence_transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

# 다국어 임베딩 모델 사용 (예시)
hanja_embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# ChromaDB를 사용하여 벡터 저장소 생성 (고유한 컬렉션 이름 지정)
hanja_db = Chroma.from_documents(ancient_docs, hanja_embedding_model, collection_name="ancient_hanja_collection")

print("한자 고문서를 위한 벡터 저장소를 성공적으로 생성했습니다.")

--- Re-creating Hanja Embedding Model and Vector Store ---


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

한자 고문서를 위한 벡터 저장소를 성공적으로 생성했습니다.


**Reasoning**:
The `hanja_db` has now been successfully created. The next logical step is to re-run the code that defines the `hanja_qa` chain, which previously failed due to the `NameError`, as per the instructions.



In [89]:
print("--- Re-creating hanja_qa RetrievalQA chain ---")

from langchain_classic.chains.retrieval_qa.base import RetrievalQA

# Retriever 설정 (기존 llm_rag 사용, 새 벡터 저장소 연결)
hanja_qa = RetrievalQA.from_chain_type(llm_rag, chain_type="stuff",
                                 retriever=hanja_db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

print("hanja_qa RetrievalQA chain successfully created.")

--- Re-creating hanja_qa RetrievalQA chain ---
hanja_qa RetrievalQA chain successfully created.


### 고문서 OCR 및 텍스트 추출의 어려움 설명

광개토대왕비문과 같이 오래되고 손상된 고문서 이미지에서 정확하게 OCR(광학 문자 인식)을 수행하고 텍스트를 추출하는 것은 매우 어려운 작업입니다.

1.  **고문서 OCR의 난이도**: 고문서는 일반적으로 종이의 노화, 훼손, 잉크 번짐, 희미해진 글자, 비정형적인 서체 등 다양한 손상을 포함하고 있습니다. 이러한 손상된 이미지에서 텍스트를 정확하게 인식하는 것은 현대의 깨끗한 문서에 비해 훨씬 복잡하고 높은 오류율을 보입니다. 일반적인 OCR 기술은 정형화된 글꼴과 깨끗한 배경을 전제로 학습된 경우가 많아, 고문서의 불규칙성과 다양성을 처리하는 데 한계가 있습니다.

2.  **EasyOCR 설정의 한계**: 현재 EasyOCR의 설정이 'ch_sim'(중국어 간체)으로 되어 있는데, 이는 한자 고문서에 적합하지 않을 수 있습니다. 고문서에 사용된 한자는 현대 중국어 간체자와는 다른 번체자 또는 더 오래된 자형을 포함하며, 문체 또한 고어체인 경우가 많습니다. EasyOCR의 기본 모델은 현대적인 글꼴과 언어 패턴에 최적화되어 있어, 고문서 특유의 자형과 문맥을 정확하게 인식하기 어렵습니다. 따라서 고문서 OCR의 정확도를 높이기 위해서는 해당 시대의 서체와 언어에 맞춰 특별히 훈련된 모델이 필요합니다.

3.  **정확도 향상을 위한 방안**: 고문서 OCR의 정확도를 높이기 위해서는 다음 단계들이 필수적입니다.
    *   **이미지 전처리**: 노이즈 제거, 대비 조절, 글자 강조 등 손상된 이미지를 개선하는 전처리 과정이 중요합니다.
    *   **전문 OCR 모델**: 특정 시대와 서체에 맞춰 훈련된 전문 OCR 모델을 사용하거나, 기존 모델을 고문서 데이터로 추가 학습(Fine-tuning)하는 방식이 필요합니다.


**Reasoning**:
The `hanja_qa` chain has been successfully created, so the next step is to execute the query and get the result, as per the instructions in the `hanja_db` NameError resolution.



### 고문서 OCR 및 텍스트 추출의 어려움 설명

광개토대왕비문과 같이 오래되고 손상된 고문서 이미지에서 정확하게 OCR(광학 문자 인식)을 수행하고 텍스트를 추출하는 것은 매우 어려운 작업입니다.

1.  **고문서 OCR의 난이도**: 고문서는 일반적으로 종이의 노화, 훼손, 잉크 번짐, 희미해진 글자, 비정형적인 서체 등 다양한 손상을 포함하고 있습니다. 이러한 손상된 이미지에서 텍스트를 정확하게 인식하는 것은 현대의 깨끗한 문서에 비해 훨씬 복잡하고 높은 오류율을 보입니다. 일반적인 OCR 기술은 정형화된 글꼴과 깨끗한 배경을 전제로 학습된 경우가 많아, 고문서의 불규칙성과 다양성을 처리하는 데 한계가 있습니다.

2.  **EasyOCR 설정의 한계**: 현재 EasyOCR의 설정이 'ch_sim'(중국어 간체)으로 되어 있는데, 이는 한자 고문서에 적합하지 않을 수 있습니다. 고문서에 사용된 한자는 현대 중국어 간체자와는 다른 번체자 또는 더 오래된 자형을 포함하며, 문체 또한 고어체인 경우가 많습니다. EasyOCR의 기본 모델은 현대적인 글꼴과 언어 패턴에 최적화되어 있어, 고문서 특유의 자형과 문맥을 정확하게 인식하기 어렵습니다. 따라서 고문서 OCR의 정확도를 높이기 위해서는 해당 시대의 서체와 언어에 맞춰 특별히 훈련된 모델이 필요합니다.

3.  **정확도 향상을 위한 방안**: 고문서 OCR의 정확도를 높이기 위해서는 다음 단계들이 필수적입니다.
    *   **이미지 전처리**: 노이즈 제거, 대비 조절, 글자 강조 등 손상된 이미지를 개선하는 전처리 과정이 중요합니다.
    *   **전문 OCR 모델**: 특정 시대와 서체에 맞춰 훈련된 전문 OCR 모델을 사용하거나, 기존 모델을 고문서 데이터로 추가 학습(Fine-tuning)하는 방식이 필요합니다.


### 웹페이지 요약 RAG 시스템 결과 검토

위 답변은 제공된 웹페이지의 내용을 RAG 시스템이 요약한 결과입니다. 답변이 웹페이지의 주요 내용을 잘 담고 있는지 확인해보세요.

### 웹페이지 요약 RAG 예제 (새로운 코드 블록)

제공된 웹페이지(`https://www.pado.kr/article/2026092513108817987`)의 내용을 WebBaseLoader를 사용하여 로드하고, RAG 시스템을 통해 요약하는 과정을 다시 살펴보겠습니다.

### 고문서 OCR 및 텍스트 추출의 어려움 설명

광개토대왕비문과 같이 오래되고 손상된 고문서 이미지에서 정확하게 OCR(광학 문자 인식)을 수행하고 텍스트를 추출하는 것은 매우 어려운 작업입니다.

1.  **고문서 OCR의 난이도**: 고문서는 일반적으로 종이의 노화, 훼손, 잉크 번짐, 희미해진 글자, 비정형적인 서체 등 다양한 손상을 포함하고 있습니다. 이러한 손상된 이미지에서 텍스트를 정확하게 인식하는 것은 현대의 깨끗한 문서에 비해 훨씬 복잡하고 높은 오류율을 보입니다. 일반적인 OCR 기술은 정형화된 글꼴과 깨끗한 배경을 전제로 학습된 경우가 많아, 고문서의 불규칙성과 다양성을 처리하는 데 한계가 있습니다.

2.  **EasyOCR 설정의 한계**: 현재 EasyOCR의 설정이 'ch_sim'(중국어 간체)으로 되어 있는데, 이는 한자 고문서에 적합하지 않을 수 있습니다. 고문서에 사용된 한자는 현대 중국어 간체자와는 다른 번체자 또는 더 오래된 자형을 포함하며, 문체 또한 고어체인 경우가 많습니다. EasyOCR의 기본 모델은 현대적인 글꼴과 언어 패턴에 최적화되어 있어, 고문서 특유의 자형과 문맥을 정확하게 인식하기 어렵습니다. 따라서 고문서 OCR의 정확도를 높이기 위해서는 해당 시대의 서체와 언어에 맞춰 특별히 훈련된 모델이 필요합니다.

3.  **정확도 향상을 위한 방안**: 고문서 OCR의 정확도를 높이기 위해서는 다음 단계들이 필수적입니다.
    *   **이미지 전처리**: 노이즈 제거, 대비 조절, 글자 강조 등 손상된 이미지를 개선하는 전처리 과정이 중요합니다.
    *   **전문 OCR 모델**: 특정 시대와 서체에 맞춰 훈련된 전문 OCR 모델을 사용하거나, 기존 모델을 고문서 데이터로 추가 학습(Fine-tuning)하는 방식이 필요합니다.

# Task
`hanja_db` was not defined, causing a `NameError`. I will re-run the cell where `hanja_db` is created to ensure it's properly initialized.

Execute the following cell:
```python
# 한자 처리에 적합한 임베딩 모델 사용 (예: 다국어 모델)
# OpenAI embedding_model은 영어에 최적화되어 있을 수 있습니다.
# !pip install -q sentence_transformers
from langchain_community.embeddings import HuggingFaceEmbeddings

# 다국어 임베딩 모델 사용 (예시)
hanja_embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")

# ChromaDB를 사용하여 벡터 저장소 생성 (고유한 컬렉션 이름 지정)
hanja_db = Chroma.from_documents(ancient_docs, hanja_embedding_model, collection_name="ancient_hanja_collection")

print("한자 고문서를 위한 벡터 저장소를 생성했습니다.")
```

## 고문서 OCR 및 텍스트 추출의 어려움 설명

### Subtask:
광개토대왕비문과 같이 손상된 고문서 이미지에서 정확하게 OCR을 수행하고 텍스트를 추출하는 것이 매우 어렵다는 점과, EasyOCR의 현재 설정(중국어 간체)의 한계를 설명합니다.


## 고문서 RAG 파이프라인의 NameError 해결 및 RAG 시스템 완성

### Subtask:
한자 고문서 RAG 파이프라인에서 발생한 `NameError: name 'hanja_db' is not defined` 오류를 해결하고, 가정한 텍스트를 사용하여 RAG 시스템을 완성합니다.


**Reasoning**:
The previous step successfully executed the `hanja_qa` query. Now, I need to display the result of this query to complete the RAG system and check its output.



### RAG 시스템의 누락 문자 복원 및 학술적 재해석 한계

RAG(Retrieval Augmented Generation) 시스템의 주요 기능은 **제공된 텍스트 데이터를 기반으로 정보를 검색하고, 이를 활용하여 사용자 질문에 대한 응답을 생성**하는 것입니다. 즉, RAG는 이미 존재하는 텍스트에서 관련성 높은 정보를 찾아내는 데 최적화되어 있습니다.

그러나 OCR(광학 문자 인식) 과정에서 발생하는 오류로 인해 **누락되거나 잘못 인식된 문자를 RAG 시스템이나 범용 LLM이 '복원'하거나 학술적으로 '재해석'하는 것은 그 범위를 넘어섭니다.** 이러한 작업은 다음과 같은 이유로 일반적인 RAG 시스템의 한계에 해당합니다:

1.  **정보 부재**: RAG 시스템은 입력된 문서 내에서만 정보를 찾고 생성합니다. 만약 OCR 오류로 인해 특정 문자가 누락되거나 잘못 인식되었다면, 시스템은 해당 '정보'를 가지고 있지 않으므로 이를 '복원'하거나 '추론'할 근거가 없습니다.
2.  **맥락 및 전문 지식 부족**: 고문서의 누락 문자를 복원하거나 학술적으로 재해석하는 것은 단순한 텍스트 매칭을 넘어섭니다. 이는 **고도의 이미지 처리 기술** (훼손된 문자 패턴 분석), **역사 언어학적 지식** (해당 시대의 문법, 어휘, 서체에 대한 이해), **그리고 해당 분야에 특화된 심층적인 모델** (예: 고문서 복원용 AI)을 요구합니다. 범용 LLM이나 RAG 시스템은 이러한 전문적인 도메인 지식과 시각적 분석 능력을 내재하고 있지 않습니다.

따라서 RAG 시스템은 'OCR 결과가 정확하다는 가정 하에' 그 텍스트에서 정보를 검색하고 응답하는 데 강점을 가지지만, OCR 자체의 오류를 수정하거나 원본 이미지의 누락된 부분을 학술적으로 재구성하는 역할은 수행하기 어렵습니다. 이러한 작업은 별도의 전문적인 전처리 과정과 특화된 AI 모델이 필요한 영역입니다.


## 현재 RAG 파이프라인의 활용 가능성 및 다음 단계 제시

### Subtask:
현재 설정된 고문서 RAG 파이프라인이 성공적으로 추출된(혹은 가정한) 고문서 텍스트를 이해하고 질문에 답하는 데 사용될 수 있음을 설명합니다. 이어서 고문서 RAG 파이프라인의 NameError를 해결하고, 현재 가정한 텍스트를 사용하여 RAG 시스템을 완성하는 단계를 진행합니다.


## Final Task

### Subtask:
RAG 파이프라인이 처리할 수 있는 범위와 한계를 명확히 이해하고, 고문서 텍스트 처리에 대한 실제적인 결과와 개선 방향을 정리합니다.


## Summary:

### Data Analysis Key Findings

*   **OCR Challenges for Ancient Documents**: Optical Character Recognition (OCR) for ancient documents like the Gwanggaeto Stele is inherently difficult due to physical damage, archaic script styles (traditional characters vs. simplified), and the limitations of general-purpose OCR tools such as EasyOCR's 'ch\_sim' (simplified Chinese) model.
*   **RAG System Limitation in Text Restoration**: The RAG (Retrieval Augmented Generation) pipeline, by its design, cannot independently restore characters missing or misrecognized due to OCR errors, nor can it perform academic reinterpretation of damaged texts. Such tasks require specialized image processing, historical linguistic expertise, and fine-tuned AI models.
*   **`NameError` Resolution**: A `NameError` for `hanja_db` was successfully resolved by re-executing the code cell responsible for initializing the `HuggingFaceEmbeddings` model (`sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2`) and creating the `Chroma` vector store named "ancient\_hanja\_collection".
*   **Functional Hanja RAG Pipeline**: After resolving the `NameError`, the Hanja RAG pipeline was successfully re-established, including the `RetrievalQA` chain, and demonstrated its ability to process queries and provide coherent answers based on given ancient Hanja text (using assumed Confucian Analects passages).

### Insights or Next Steps

*   **Pre-process Ancient Document Images**: Implement advanced image pre-processing techniques (e.g., noise reduction, contrast adjustment, character enhancement) to significantly improve the quality of input for OCR, thereby increasing the accuracy of text extraction from damaged ancient documents.
*   **Utilize Specialized OCR Models**: For real-world ancient document processing, adopt or develop specialized OCR models trained on traditional and archaic Hanja scripts, potentially including fine-tuning existing models with specific historical document datasets, instead of relying on models optimized for modern simplified Chinese.
